<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 자기지도 학습 · 멀티모달 · 02. SimCLR: augmentation과 NT-Xent

## A Simple Framework for Contrastive Learning of Visual Representations

- **원 논문:** [A Simple Framework for Contrastive Learning of Visual Representations](https://arxiv.org/abs/2002.05709)
- **저자 / 발표:** Ting Chen, Simon Kornblith, Mohammad Norouzi, Geoffrey Hinton · ICML (2020)
- **난이도 / 예상 시간:** 중급 · 약 70분
- **선수 지식:** cosine similarity, cross entropy, augmentation
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 두 augmentation view, encoder f, projection g, 대칭 NT-Xent를 구현합니다.

**축소하거나 생략한 부분:** ImageNet·ResNet-50·batch 4096 대신 96개 8×8 이미지와 작은 MLP를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | Figure 2 | x→두 view→f→g | view 비동일성·shape |
| §2.1, Eq. (1) | L2-normalized cosine | 대각/범위 |
| §2.1, Eq. (2) | 2N 대칭 NT-Xent | self mask·positive index |
| Algorithm 1 및 §2.3 | projection 학습·h 평가 | same/different class cosine |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\ell_{i,j}=-\log\frac{\exp(\operatorname{sim}(z_i,z_j)/\tau)}{\sum_{k\ne i}\exp(
\operatorname{sim}(z_i,z_k)/\tau)}$$

- **기호 정의:** i와 j는 같은 샘플의 두 view, z는 projection, τ는 온도입니다.
- **수식의 역할:** 강한 augmentation pair를 positive로 두고 2B-2개 view를 negative로 사용합니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 Figure 2을 구현합니다.
- **입출력 shape:** 두 view [B, 1, 8, 8] → projection [2B, D] → similarity [2B, 2B]
- **평가:** 대칭 NT-Xent와 같은 클래스/다른 클래스 cosine gap
- **원문 대비 한계:** ImageNet·ResNet-50·batch 4096 대신 96개 8×8 이미지와 작은 MLP를 사용합니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** Figure 2
- **이 셀의 책임:** x→두 view→f→g
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** view 비동일성·shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 02-1: 서로 독립적인 두 brightness/noise view를 만드세요.
def make_views(x: Tensor):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def aug():
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2.1, Eq. (1) / §2.1, Eq. (2)
- **이 셀의 책임:** L2-normalized cosine / 2N 대칭 NT-Xent
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 대각/범위 / self mask·positive index. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 02-2: Eq. (1)–(2)의 2N NT-Xent를 구현하세요.
def nt_xent(z1: Tensor, z2: Tensor, temperature=0.2):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** Figure 2 / §2.1, Eq. (1) / §2.1, Eq. (2) / Algorithm 1 및 §2.3
- **이 셀의 책임:** x→두 view→f→g / L2-normalized cosine / 2N 대칭 NT-Xent / projection 학습·h 평가
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** view 비동일성·shape / 대각/범위 / self mask·positive index / same/different class cosine. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 02-3: encoder f/projector g를 학습하고 h의 class cosine을 그리세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** Figure 2 / §2.1, Eq. (1) / §2.1, Eq. (2)
- **이 셀의 책임:** x→두 view→f→g / L2-normalized cosine / 2N 대칭 NT-Xent
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** view 비동일성·shape / 대각/범위 / self mask·positive index. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO P-1: 논문의 modality 준비, 핵심 module, objective와 평가를 구현하세요.
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    hidden_dim: int = 16
    temperature: float = 0.2
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(batch: Tensor) -> tuple[Tensor, Tensor]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


class SimclrPortfolioModel(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, config: PortfolioConfig):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, view: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def nt_xent_logits(
        self, first_projection: Tensor, second_projection: Tensor
    ) -> tuple[Tensor, Tensor]:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def compute_loss(self, first: Tensor, second: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def training_step(self, batch: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §2.1, Eq. (2)
- **이 셀의 책임:** 2N 대칭 NT-Xent
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** self mask·positive index. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: SimclrPortfolioModel, batch: Tensor) -> list[float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §2.1, Eq. (2)
- **이 셀의 책임:** 2N 대칭 NT-Xent
- **Tensor shape 계약:** 두 view [B,C,H,W] → z₁/z₂ [B,D] → similarity [2B,2B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** self mask·positive index. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: SimclrPortfolioModel, batch: Tensor
) -> dict[str, float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.